In [1]:
import os
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np


def build_df_from_folder(folder_path, min_white_ratio=0.03, verbose=True):
    """
    Scans a folder structure to build a DataFrame of valid image-mask pairs.

    Args:
        folder_path (str): Path to the root folder.
        min_white_ratio (float): Minimum ratio of white pixels to keep the mask.
        verbose (bool): Whether to print details about skipped pairs.

    Returns:
        pd.DataFrame: DataFrame with columns ['image_path', 'mask_path']
    """
    data = []
    skipped = 0
    total = 0

    folder_path = os.path.abspath(folder_path)

    for subfolder in tqdm(sorted(os.listdir(folder_path)), desc="Scanning subfolders"):
        sub_path = os.path.join(folder_path, subfolder)
        if os.path.isdir(sub_path):
            files = sorted(os.listdir(sub_path))
            image_files = [f for f in files if f.endswith('_image.png')]

            for img_file in image_files:
                prefix = img_file.replace('_image.png', '')
                img_path = os.path.join(sub_path, f"{prefix}_image.png")
                mask_path = os.path.join(sub_path, f"{prefix}_mask.png")
                total += 1

                if not os.path.exists(mask_path):
                    if verbose:
                        print(f"[Missing] {mask_path}")
                    skipped += 1
                    continue

                # Load and analyze mask
                mask = Image.open(mask_path).convert("L")
                mask_array = np.array(mask)
                white_ratio = np.sum(mask_array > 0) / mask_array.size

                if white_ratio < min_white_ratio:
                  #  if verbose:
                     #   print(f"[Skipped] Low foreground ratio ({white_ratio:.4f}) in: {mask_path}")
                    skipped += 1
                    continue

                data.append((img_path, mask_path))

    df = pd.DataFrame(data, columns=["image_path", "mask_path"])
    print(f"\n{folder_path}")
    print(f"Valid Pairs: {len(df)}")
    print(f"Skipped (missing/empty/low foreground): {skipped} out of {total}")
    return df

In [2]:
# --- Helpers for TFRecord ---
def _bytes_feature(value):
    return tf.train.Feature(bytes_list=tf.train.BytesList(value=[value]))

def serialize_example(image, mask):
    def encode_img(img):
        with io.BytesIO() as output:
            img.save(output, format="PNG")
            return output.getvalue()
    image_bytes = encode_img(image)
    mask_bytes = encode_img(mask)
    feature = {
        'image': _bytes_feature(image_bytes),
        'mask': _bytes_feature(mask_bytes),
    }
    example = tf.train.Example(features=tf.train.Features(feature=feature))
    return example.SerializeToString()

In [3]:
# --- Write TFRecord ---
def write_tfrecord(df, out_path, log_title=""):
    with tf.io.TFRecordWriter(out_path) as writer:
        for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Writing {log_title}"):
            image = Image.open(row['image_path']).convert("L")
            mask = Image.open(row['mask_path']).convert("L")
            example = serialize_example(image, mask)
            writer.write(example)
    print(f"Saved {log_title} to {out_path}")

In [4]:
import io
# --- Main ---
base_dir = "/kaggle/input/ddsmtrainvaldataset/crops_output"
output_dir = "/kaggle/working/tfrecords"
os.makedirs(output_dir, exist_ok=True)

# Build dataframes
train_dir = os.path.join(base_dir, "train")
val_dir = os.path.join(base_dir, "val")

train_df = build_df_from_folder(train_dir)
val_df = build_df_from_folder(val_dir)

# Write TFRecords
write_tfrecord(train_df, os.path.join(output_dir, "train.tfrecord"), "Train")
write_tfrecord(val_df, os.path.join(output_dir, "val.tfrecord"), "Validation")

Scanning subfolders: 100%|██████████| 1085/1085 [11:13<00:00,  1.61it/s]



/kaggle/input/ddsmtrainvaldataset/crops_output/train
Valid Pairs: 14291
Skipped (missing/empty/low foreground): 44304 out of 58595


Scanning subfolders: 100%|██████████| 233/233 [02:31<00:00,  1.54it/s]



/kaggle/input/ddsmtrainvaldataset/crops_output/val
Valid Pairs: 3036
Skipped (missing/empty/low foreground): 9287 out of 12323


Writing Train: 100%|██████████| 14291/14291 [07:10<00:00, 33.20it/s]


Saved Train to /kaggle/working/tfrecords/train.tfrecord


Writing Validation: 100%|██████████| 3036/3036 [01:33<00:00, 32.63it/s]

Saved Validation to /kaggle/working/tfrecords/val.tfrecord


In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np

# Path to your TFRecord file (update this if needed)
tfrecord_path = "/kaggle/working/tfrecords/train.tfrecord"

# 1. Define parser for TFRecord
def parse_example(example_proto):
    feature_description = {
        'image': tf.io.FixedLenFeature([], tf.string),
        'mask': tf.io.FixedLenFeature([], tf.string),
    }
    parsed = tf.io.parse_single_example(example_proto, feature_description)

    image = tf.io.decode_png(parsed['image'], channels=1)
    mask = tf.io.decode_png(parsed['mask'], channels=1)

    # Normalize for display (optional)
    image = tf.image.convert_image_dtype(image, tf.float32)
    mask = tf.image.convert_image_dtype(mask, tf.float32)

    return image, mask

# 2. Load and parse TFRecord
raw_dataset = tf.data.TFRecordDataset(tfrecord_path)
parsed_dataset = raw_dataset.map(parse_example)

# 3. Display first 5 image–mask pairs
plt.figure(figsize=(10, 5))
for i, (image, mask) in enumerate(parsed_dataset.take(5)):
    img_np = image.numpy().squeeze()
    mask_np = mask.numpy().squeeze()

    plt.subplot(2, 5, i + 1)
    plt.imshow(img_np, cmap="gray")
    plt.title(f"Image {i+1}")
    plt.axis("off")

    plt.subplot(2, 5, i + 6)
    plt.imshow(mask_np, cmap="gray")
    plt.title(f"Mask {i+1}")
    plt.axis("off")

plt.tight_layout()
plt.show()